01. Project Vision

02. Problem Statement

03. Understanding the Dataset

04. Data Preparation

05. Why Neural Networks?

06. Building Intuition

07. Mathematics Behind ANN

08. Forward Propagation

09. Activation Functions

10. Loss Functions

11. Backpropagation

12. Gradient Descent

13. Building First ANN

14. Training

15. Evaluation

16. Model Improvements

17. Hyperparameter Experiments

18. Visualization of Learning

19. Comparing ML vs ANN

20. Final Conclusions

21. Future Experiments

In [3]:
import pandas as pd

DATA_PATH = "../data/train.csv"

df = pd.read_csv(DATA_PATH, sep=";")

print(f"Number of rows    : {df.shape[0]}")
print(f"Number of columns : {df.shape[1]}")

Number of rows    : 45211
Number of columns : 17


In [4]:
# Standardize column names to be more descriptive and self-explanatory
df.rename(columns={
    "y": "target",
    "default": "default_credit",
    "housing": "housing_loan",
    "loan": "personal_loan",
    "contact": "contact_type",
    "poutcome": "previous_outcome",
    "pdays": "no_time_contacted_days_before",
    "duration": "last_call_duration",
    "campaign": "contacts_in_campaign",
    "previous": "contacted_in_before_campaing",
}, inplace=True)

# Recompute column groups after renaming
num_col = df.select_dtypes(include=["int64", "float64"]).columns
cat_col = df.select_dtypes(include=["object"]).columns

print("Numerical columns   :", list(num_col))
print("Categorical columns :", list(cat_col))

Numerical columns   : ['age', 'balance', 'day', 'last_call_duration', 'contacts_in_campaign', 'no_time_contacted_days_before', 'contacted_in_before_campaing']
Categorical columns : ['job', 'marital', 'education', 'default_credit', 'housing_loan', 'personal_loan', 'contact_type', 'month', 'previous_outcome', 'target']


C:\Users\midhu\AppData\Local\Temp\ipykernel_34764\3205276823.py:17: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_col = df.select_dtypes(include=["object"]).columns


In [5]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [6]:
# preprocessing_pipeline is defined in preprocessing.py, which wires together the
# ColumnTransformer above with the FeatureSelector from custom_transformer.py
import sys
import os

# Add src/ to Python's import search path (notebooks/ and src/ are siblings)
sys.path.append(os.path.abspath(os.path.join("..", "src")))

from preprocessing import preprocessing_pipeline

In [7]:
preprocessing_pipeline

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators <combining_estimators>` for more details.","[('preprocessor', ...), ('feature_selector', ...)]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing <metadata_routing>`.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('log_num', ...), ...]"
,"remainder remainder: {'drop', 'passthrough'} or estimator, default='drop'By default, only the specified columns in `transformers` aretransformed and combined in the output, and the non-specifiedcolumns are dropped. (default of ``'drop'``).By specifying ``remainder='passthrough'``, all remaining columns thatwere not specified in `transformers`, but present in the data passedto `fit` will be automatically passed through. This subset of columnsis concatenated with the output of the transformers. For dataframes,extra columns not seen during `fit` will be excluded from the outputof `transform`.By setting ``remainder`` to be an estimator, the remainingnon-specified columns will use the ``remainder`` estimator. Theestimator must support :term:`fit` and :term:`transform`.Note that using this feature requires that the DataFrame columnsinput at :term:`fit` and :term:`transform` have identical order.",'drop'
,"sparse_threshold sparse_threshold: floa

In [8]:
from sklearn.model_selection import train_test_split

# Use the ORIGINAL raw column names (pre-rename) since preprocessing_pipeline expects
# the raw UCI schema — this keeps the pipeline reusable directly in train.py / predict.py
RAW_DATA_PATH = "../data/train.csv"
raw_df = pd.read_csv(RAW_DATA_PATH, sep=";")

X = raw_df.drop(columns=["y"])
y = raw_df["y"].map({"yes": 1, "no": 0})

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y,   # preserve the ~88/12 class balance in both splits
)

print(f"X_train: {X_train.shape}   X_test: {X_test.shape}")
print(f"y_train positive rate: {y_train.mean():.4f}   y_test positive rate: {y_test.mean():.4f}")

X_train: (36168, 16)   X_test: (9043, 16)
y_train positive rate: 0.1170   y_test positive rate: 0.1170


In [7]:
# Fit the preprocessing pipeline on the training data ONLY, then transform both splits.
# fit_transform learns scaler means/std + encoder categories from X_train;
# transform (no fit) reuses those same learned statistics on X_test.
X_train_processed = preprocessing_pipeline.fit_transform(X_train)
X_test_processed = preprocessing_pipeline.transform(X_test)

print("Processed feature columns:", list(X_train_processed.columns))
print("Processed shapes -> train:", X_train_processed.shape, " test:", X_test_processed.shape)

X_train_processed.head()

Processed feature columns: ['age', 'education', 'housing_loan', 'personal_loan', 'contacts_in_campaign', 'contacted_in_before_campaing', 'balance_log', 'duration_log', 'job_blue-collar', 'job_student', 'marital_married', 'marital_single', 'previous_outcome_other', 'previous_outcome_success', 'previous_outcome_unknown', 'contact_type_telephone', 'contact_type_unknown']
Processed shapes -> train: (36168, 17)  test: (9043, 17)


,age,education,housing_loan,personal_loan,contacts_in_campaign,contacted_in_before_campaing,balance_log,duration_log,job_blue-collar,job_student,marital_married,marital_single,previous_outcome_other,previous_outcome_success,previous_outcome_unknown,contact_type_telephone,contact_type_unknown
0,-0.460434,2.0,0.0,0.0,-0.246104,-0.241509,-0.127042,-0.241798,0.0,0.0,0.0,0.0,0.0,0.0,1.0,1.0,0.0
1,-1.589641,2.0,0.0,0.0,0.398202,2.664584,1.338886,1.776444,0.0,1.0,0.0,1.0,0.0,0.0,0.0,0.0,0.0
2,0.292371,2.0,1.0,0.0,0.398202,-0.241509,-0.464179,2.478738,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0
3,0.668773,2.0,0.0,0.0,2.653271,-0.241509,-0.604502,-1.721212,0.0,0.0,1.0,0.0,0.0,0.0,1.0,1.0,0.0
4,-0.272233,2.0,0.0,0.0,2.331118,-0.241509,-0.456820,-1.204404,0.0,0.0,1.0,0.0,0.0,0.0,1.0,0.0,0.0


## Building the base model 

In [1]:
import tensorflow as tf
import keras
import numpy as np
import pandas as pd
import sklearn
import cv2
import matplotlib.pyplot as plt

print("TensorFlow :", tf.__version__)
print("Keras      :", keras.__version__)
print("NumPy      :", np.__version__)
print("Pandas     :", pd.__version__)
print("Scikit     :", sklearn.__version__)
print("OpenCV     :", cv2.__version__)

TensorFlow : 2.21.0
Keras      : 3.15.0
NumPy      : 2.4.6
Pandas     : 3.0.3
Scikit     : 1.9.0
OpenCV     : 5.0.0


## Step 1 : Design Study 

In [11]:
print(X_train.shape)
print(X_test.shape)
print(y_train.shape)
print(y_test.shape)

print(X_train.columns)

(36168, 16)
(9043, 16)
(36168,)
(9043,)
Index(['age', 'job', 'marital', 'education', 'default', 'balance', 'housing',
       'loan', 'contact', 'day', 'month', 'duration', 'campaign', 'pdays',
       'previous', 'poutcome'],
      dtype='str')


In [9]:
from tensorflow.keras.models import Sequential

In [10]:
# intializing the model 
model = Sequential()

# input layer
model.add(
    Dense(
        units=32,
        activation="relu",
        input_shape=(16,)
    )
)
# hidden layer 
